# Fine-tune Qwen2.5-Coder-7B with Unsloth (QLoRA)
Runs on free Colab T4 (16GB). Upload your `dataset.jsonl` (from prepare_dataset.py) before running.

Steps: install -> load base model in 4-bit -> attach LoRA -> train -> export GGUF (q4_k_m) for Ollama.

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# Upload dataset.jsonl (from prepare_dataset.py) via the Colab file browser first.
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for instruction, inp, output in zip(examples["instruction"], examples["input"], examples["output"]):
        text = alpaca_prompt.format(instruction, inp, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()

## Export to GGUF for Ollama deployment

In [ ]:
model.save_pretrained_gguf("my-coder-model", tokenizer, quantization_method="q4_k_m")
# This produces my-coder-model/*.gguf -- download it (or push to Hugging Face Hub below)

In [ ]:
# Optional: push GGUF straight to your Hugging Face account so RunPod/Ollama can pull it directly
# from huggingface_hub import login
# login(token="hf_...")
# model.push_to_hub_gguf("YOUR_HF_USERNAME/my-coder-model", tokenizer, quantization_method="q4_k_m")